# 07 · 動畫：讓圖動起來（FuncAnimation）

前六課畫的都是靜態圖。這課讓圖**動起來**——用 `FuncAnimation` 做動畫，在 Colab 裡直接播放，還能存成 GIF 分享。

> 💡 可執行 notebook，`Shift+Enter` 跑每一格。動畫格子跑完下方會出現播放控制列。

## 學習目標

- 理解 `FuncAnimation` 的核心：一個**每幀被呼叫的 `update` 函式**
- 搞懂 `frames` / `interval` / `blit` 三個關鍵參數
- 在 Colab 用 `to_jshtml()` 內嵌播放（不需要 ffmpeg）
- 把動畫**存成 GIF**

## 1. 動畫的核心概念

動畫 = 一連串靜態畫格快速播放。matplotlib 的做法是：你先畫好第一張圖、拿到要更新的物件（例如那條線），然後寫一個 `update(frame)` 函式描述「第 frame 幀該長怎樣」。`FuncAnimation` 會幫你一幀一幀呼叫它。

關鍵：**不要每幀重畫整張圖，而是更新既有物件的資料**（用 `line.set_ydata(...)`），這樣才快。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

x = np.linspace(0, 2 * np.pi, 200)

fig, ax = plt.subplots(figsize=(7, 4))
line, = ax.plot(x, np.sin(x), color="tab:blue")   # 注意逗號：plot 回傳 list，取出那條線
ax.set_ylim(-1.3, 1.3)
ax.set_title("A traveling wave")

def update(frame):
    # 每幀把相位往右推一點，看起來就像波在移動
    line.set_ydata(np.sin(x + frame * 0.1))
    return (line,)            # blit=True 時要回傳「這幀改動的物件」

ani = FuncAnimation(fig, update, frames=120, interval=40, blit=True)

plt.close(fig)               # 關掉靜態圖，只留下動畫（否則 notebook 會多顯示一張）
HTML(ani.to_jshtml())        # 在 Colab 內嵌成可播放的 HTML（純 JS，免裝 ffmpeg）

## 2. 三個關鍵參數

| 參數 | 意思 |
| --- | --- |
| `frames` | 總幀數（給整數）或每幀要傳給 `update` 的值（給序列） |
| `interval` | 每幀間隔**毫秒**，`interval=40` ≈ 25 fps |
| `blit` | `True` 只重畫有變動的部分，較快；此時 `update` 要 `return` 改動的物件 |

⚠️ **一定要把 `FuncAnimation` 的結果存進變數**（這裡是 `ani`）。如果不存，物件會被垃圾回收，動畫就動不了。

**動手試試**：把 `interval` 改成 `100`（變慢）、`frames` 改成 `60`（變短），重跑感受差別。

## 3. 逐步「畫出」一條曲線

另一個常見手法：每幀多畫一點點，看起來就像曲線被一筆一筆描出來。這裡畫一條 Lissajous 曲線。

In [ ]:
t = np.linspace(0, 2 * np.pi, 300)
xs, ys = np.sin(3 * t), np.sin(2 * t)

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_title("Drawing a Lissajous curve")

line, = ax.plot([], [], color="tab:red", lw=2)        # 一開始是空的
dot, = ax.plot([], [], "o", color="tab:red")          # 目前筆尖

def update(i):
    line.set_data(xs[:i], ys[:i])      # 畫到第 i 點
    dot.set_data([xs[i - 1]], [ys[i - 1]])
    return line, dot

ani = FuncAnimation(fig, update, frames=range(1, len(t)), interval=20, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

## 4. 存成 GIF

用 `PillowWriter` 就能存 GIF，**不需要 ffmpeg**（Pillow 是 matplotlib 常見的相依套件）。存完可以下載分享，或 `![](檔名.gif)` 內嵌。

In [ ]:
from matplotlib.animation import PillowWriter

ani.save("lissajous.gif", writer=PillowWriter(fps=30))
print("已存成 lissajous.gif")

# 在 Colab 想下載：
# from google.colab import files
# files.download("lissajous.gif")

> 想存成 mp4（`ani.save("out.mp4")`）需要系統有 **ffmpeg**；Colab 可先 `!apt-get -qq install ffmpeg`。GIF 通常已足夠分享用。

## 小結

- 動畫 = 先畫好物件，再用 `update(frame)` **更新物件資料**（不是重畫整張圖）。
- `FuncAnimation(fig, update, frames=..., interval=..., blit=True)`，記得**把結果存進變數**。
- Colab 內嵌播放用 `HTML(ani.to_jshtml())`；存檔用 `ani.save(..., writer=PillowWriter(fps=...))`。

## 練習

1. 做一個「彈跳球」動畫：球在框內上下彈，每幀更新 y 座標。
2. 把第 1 節的波改成**同時**畫 sin 和 cos 兩條線一起移動（更新兩個物件）。
3. 把你的動畫存成 GIF。

🎉 這是 matplotlib 模組的進階專題課。你現在能畫靜態圖、排版、調樣式，還能讓圖動起來——工具箱很完整了！